In [12]:
pip install ollama

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [13]:
import ollama, json, re
OLLAMA_HOST = "http://10.10.80.99:4001"
MODEL = "gpt-oss:120b"
client = ollama.Client(host=OLLAMA_HOST)

def safe_json_request(prompt):
    response = client.chat(model=MODEL, messages=[{"role": "user", "content": prompt}])
    content = response['message']['content']
    content = re.sub(r'```json\s*|\s*```', '', content)
    content = re.sub(r'```\s*|\s*```', '', content)
    start, end = content.find('{'), content.rfind('}') + 1
    if start == -1: start, end = content.find('['), content.rfind(']') + 1
    return json.loads(content[start:end]) if start != -1 else {}

print("✅ System Ready: Agnostic Ingestion Helper Active.")

✅ System Ready: Agnostic Ingestion Helper Active.


In [14]:
raw_artifact_blob = {
    "_key": "lib_cs_credential_dump",
    "name": "Dump Credentials (Mimikatz)",
    "tactic": "TA0006",
    "category": "Cobalt Strike",
    "description": "Extract credentials from LSASS memory using Mimikatz",
    "riskLevel": "critical",
    "cobaltStrikeCommand": "mimikatz ${command}",
    "parameters": [
        {"id": "command", "default": "sekurlsa::logonpasswords", "options": ["sekurlsa::logonpasswords", "lsadump::sam"]}
    ],
    "inputs": [{"id": "beacon", "type": "Agent_Session", "required": True}],
    "outputs": [{"id": "credentials", "type": "Credential_Set"}]
}

# AGNOSTIC USER SETTINGS
# These fields are required by Operator's NodePalette.tsx to render the sidebar
user_settings = {
    "artifact_type": "LibraryModule",
    "metadata_for_graph": ["name", "tactic", "category", "riskLevel", "description"],
    "summary_instruction": "Summarize the offensive impact for an operator."
}

In [9]:
def step_2_sidebar_metadata(data, settings):
    prompt = f"""
    OBJECT DATA: {json.dumps(data)}
    REQUIRED METADATA: {settings['metadata_for_graph']}
    
    TASK:
    1. Extract values for REQUIRED METADATA. 
    2. Provide a 1-sentence 'node_summary'.
    3. Suggest a single emoji 'icon' representing this action.
    
    Return ONLY JSON with keys: 'metadata', 'node_summary', 'icon'.
    """
    return safe_json_request(prompt)

sidebar_metadata = step_2_sidebar_metadata(raw_artifact_blob, user_settings)

In [10]:
def step_4_structure_payload(original, metadata_keys):
    prompt = f"""
    Refactor the following data into a 'Machine-Ready Execution Envelope'.
    
    ORIGINAL: {json.dumps(original)}
    EXCLUDE: {metadata_keys}
    
    STRUCTURE REQUIREMENTS:
    - 'parameters': Array of objects (id, label, type, required, default, options)
    - 'io_ports': Object containing 'inputs' and 'outputs' (id, label, type)
    - 'execution': Object containing 'type' (e.g. cobalt_strike) and the raw 'command'
    - 'metadata': Any remaining tags or versions.

    Return ONLY JSON.
    """
    return safe_json_request(prompt)

canvas_payload = step_4_structure_payload(raw_artifact_blob, user_settings['metadata_for_graph'])

In [11]:
artifact_id = raw_artifact_blob["_key"]

# WHAT GOES INTO ARANGODB (The LibraryModule Collection)
# This matches the 'LibraryModule' interface in your TypeScript
arango_node = {
    "_key": artifact_id,
    "id": artifact_id,
    "name": sidebar_metadata['metadata'].get('name'),
    "icon": sidebar_metadata.get('icon', '🛠️'),
    "tactic": sidebar_metadata['metadata'].get('tactic'),
    "category": sidebar_metadata['metadata'].get('category'),
    "riskLevel": sidebar_metadata['metadata'].get('riskLevel'),
    "description": sidebar_metadata['node_summary'],
    "payload_url": f"/api/payloads/{artifact_id}.json" # Pointer to the heavy data
}

# WHAT GOES INTO THE JSON STORE (The "Guts")
# This is what CollapsiblePropertiesPanel.tsx loads when a node is selected
heavy_payload = {
    "parameters": canvas_payload.get('parameters', []),
    "inputs": canvas_payload.get('io_ports', {}).get('inputs', []),
    "outputs": canvas_payload.get('io_ports', {}).get('outputs', []),
    "execution": canvas_payload.get('execution', {})
}

print("✅ INGESTION COMPLETE")
print("\n--- [ARANGODB: LibraryModule Document] ---")
print(json.dumps(arango_node, indent=2))

print("\n--- [STORAGE: Payload JSON] ---")
print(json.dumps(heavy_payload, indent=2))

✅ INGESTION COMPLETE

--- [ARANGODB: LibraryModule Document] ---
{
  "_key": "lib_cs_credential_dump",
  "id": "lib_cs_credential_dump",
  "name": "Dump Credentials (Mimikatz)",
  "icon": "\ud83d\udddd\ufe0f",
  "tactic": "TA0006",
  "category": "Cobalt Strike",
  "riskLevel": "critical",
  "description": "Uses Mimikatz via Cobalt Strike to dump credentials from LSASS memory.",
  "payload_url": "/api/payloads/lib_cs_credential_dump.json"
}

--- [STORAGE: Payload JSON] ---
{
  "parameters": [
    {
      "id": "command",
      "label": "command",
      "type": "string",
      "required": true,
      "default": "sekurlsa::logonpasswords",
      "options": [
        "sekurlsa::logonpasswords",
        "lsadump::sam"
      ]
    }
  ],
  "inputs": [
    {
      "id": "beacon",
      "label": "beacon",
      "type": "Agent_Session"
    }
  ],
  "outputs": [
    {
      "id": "credentials",
      "label": "credentials",
      "type": "Credential_Set"
    }
  ],
  "execution": {
    "type": "